In [1]:
import torch
import os
os.chdir("/home/hice1/stekin6/scratch/rl-vqa")
from configs import hf_token, HF_CACHE
os.environ['HF_HOME'] = HF_CACHE


dataset_name = "okvqa"
ens_result_dict = torch.load(f"results/ensemble/{dataset_name}/exp_result.pth", weights_only=False)


The history saving thread hit an unexpected error (OperationalError('disk I/O error')).History will not be written to the database.


In [3]:
import numpy as np

from data_generator.inference_loader import load_infer_prob_data
from ens_pruning.ens_methods import voting

# dataset_name = "mmmu" # okvqa
model_names = ens_result_dict["model_names"]
split_type = "validation" if dataset_name in ["mmmu", "okvqa"] else "test"
data = load_infer_prob_data(model_names, dataset_name, split_type)
label = data[:, -1]

model_probs = np.split(data[:, :-1], len(model_names), axis=1)
model_preds = []
for i in range(len(model_names)):
    print(np.mean(model_probs[i].argmax(1) == label))
    model_preds.append(model_probs[i].argmax(1))

model_preds = np.array(model_preds)
voting_ens = voting(model_preds.T, "plurality")

print("Plurality: ", np.mean(model_preds == label))

/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


0.834061135371179
0.8724890829694323
0.8532751091703057
Plurality:  0.8532751091703057


In [84]:
import numpy as np
import pandas as pd
import os
os.chdir("/home/hice1/stekin6/scratch/rl-vqa")

from model_helper import calc_metric



model_names = ["llava-v1.6-vicuna-7b-hf", "llava-v1.6-vicuna-13b-hf",
                "Qwen2.5-VL-7B-Instruct", "deepseek-vl2-tiny", "deepseek-vl2-small"]
# model_names = ["deepseek-vl2-tiny", "deepseek-vl2-small"]

inputs = []
answers = []
total_arr = []
candidates = []
for i, mn in enumerate(model_names):
    data_path = os.path.join("results", "inference", "mmmu", "validation", f"{mn}_output.csv")
    data_df = pd.read_csv(data_path)
    if i == 0:
        inputs = data_df["question"].values
        answers = data_df["answer"].values
    
    candidates.append(data_df["generated_outputs"].str.strip().values)
    print(np.mean(answers == data_df["generated_outputs"].str.strip().values))
    # scores, score_arr = calc_metric(data_df["answer"].values, data_df["generated_outputs"].values, return_scores=True)
    # total_arr.append(score_arr)
    # print(mn, scores)


0.30062111801242236
0.34658385093167704
0.0
0.18509316770186335
0.31925465838509315


array([ True,  True,  True, ...,  True,  True,  True])

In [28]:
answers

array(['B', 'C', 'B', 'D', 'B', 'A', 'B', 'A', 'C', 'A', 'A', 'A', 'C',
       'B', 'A', 'C', 'C', 'D', 'C', 'B', 'A', 'B', 'B', 'A', 'B', 'A',
       'C', 'A', 'B', 'A', 'C', 'E', 'A', 'E', 'C', 'A', 'E', 'B', 'D',
       'B', 'B', 'A', 'C', 'A', 'B', 'D', 'E', 'C', 'D', 'E', 'B', 'B',
       'D', 'A', 'E', 'D', 'E', 'D', 'C', 'A', 'A', 'A', 'C', 'C', 'A',
       'A', 'C', 'B', 'C', 'B', 'C', 'B', 'C', 'B', 'D', 'B', 'A', 'B',
       'A', 'B', 'D', 'A', 'D', 'D', 'B', 'B', 'C', 'C', 'B', 'C', 'D',
       'D', 'D', 'A', 'C', 'D', 'B', 'C', 'A', 'A', 'C', 'B', 'B', 'B',
       'A', 'D', 'D', 'C', 'B', 'A', 'B', 'C', 'A', 'A', 'B', 'B', 'B',
       'C', 'B', 'A', 'B', 'A', 'A', 'B', 'B', 'B', 'B', 'C', 'C', 'D',
       'D', 'B', 'B', 'D', 'B', 'C', 'C', 'B', 'D', 'B', 'A', 'D', 'C',
       'C', 'B', 'D', 'A', 'A', 'B', 'B', 'D', 'A', 'B', 'B', 'C', 'A',
       'A', 'A', 'A', 'A', 'B', 'B', 'E', 'A', 'B', 'A', 'C', 'A', 'C',
       'B', 'D', 'D', 'A', 'B', 'B', 'B', 'I', 'A', 'E', 'B', 'A

In [85]:
import json


with open("test_outputs_single_llava-v1.6-vicuna-7b-hf.json", "r") as f:
    lava13b_outs = json.load(f)


In [86]:
lava13b_arr = np.array([txt.split("ASSISTANT: ")[-1][0] for txt in lava13b_outs])

In [87]:
lava13b_arr

array(['A', 'B', 'B', 'B', 'A', 'B', 'A', 'A', 'A', 'B', 'B', 'B', 'B',
       'B', 'A', 'B', 'B', 'B', 'A', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'A', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'A', 'A', 'A', 'A', 'A', 'A',
       'B', 'B', 'A', 'A', 'B', 'A', 'A', 'A', 'B', 'A', 'A', 'A', 'A',
       'A', 'B', 'A', 'B', 'A', 'B', 'B', 'B', 'A', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'A', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B',
       'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B', 'B

In [88]:
np.mean(np.array(answers) == lava13b_arr)

np.float64(0.2906832298136646)

In [22]:
scores, score_arr = calc_metric(np.array(answers), lava13b_arr, return_scores=True)

In [23]:
scores

[np.float64(0.9990046053025059),
 np.float64(0.9316894368543819),
 np.float64(0.9990606445025147)]

In [51]:
data_df["generated_outputs"].values

array(['A', 'B', 'A', 'B',
       "The following data in the following the data in process the the beginning work units of for the number of units for the process is: process starting to process of finished the of the the is: for the process is: unit's of the for the process: for the process process: for the process: for the process: for the is: process: of process: process: process: process process: for process: in process: of the process: process: for process: process: process",
       'B', 'D', 'B', 'C', 'B', 'C', 'A', 'C', 'A', 'C', 'A', 'B', 'D',
       'C', 'A', 'D', 'C', 'A', 'C', 'D', 'A', 'B', 'A', 'B', 'A', 'A',
       'B', 'A', 'A',
       'neemilia sonchuria baccinia nevadella[[18492113, 90, 160, 49948]]',
       'D', 'A', nan, 'B', 'B', 'A', 'C', 'D', 'A', 'A', 'D', 'B', 'C',
       'D', 'D',
       'kidney bean[[10, 0, 6791, 2343, 985033, 18231, 21916, 7,29,65',
       'D', 'D', 'A', 'E', 'E', 'B', 'B', 'C', 'C', 'D', 'A', 'D', 'B',
       'B', 'A', 'B', 'B', 'A', 'B', 'B

In [37]:
candidates = np.array(candidates).T

In [42]:
candidates

array([['D', '(D) thirty', 'B', ..., 'B', 'B', 'A'],
       ['D', 'D', 'C', ..., 'C', 'D', 'A']], dtype=object)

In [59]:
import spacy
from sklearn.metrics.pairwise import cosine_similarity


nlp = spacy.load("en_core_web_lg")
candidates = np.array(candidates).T

outputs = []
for i in range(len(candidates)):
    question_vector = nlp(inputs[i]).vector
    pred_1 = str(candidates[i][2]).strip()
    pred_2 = str(candidates[i][3]).strip()
    vector_1 = nlp(pred_1).vector
    vector_2 = nlp(pred_2).vector

    score_1 = cosine_similarity(question_vector[None, :], vector_2[None, :]).squeeze()
    score_2 = cosine_similarity(question_vector[None, :], vector_1[None, :]).squeeze()
    if score_1 > score_2:
        outputs.append(pred_1)
    else:
        outputs.append(pred_2)

In [61]:
outputs

['D',
 'D',
 'C',
 'D',
 'B',
 'A',
 'C',
 'B',
 'A',
 'D',
 'A',
 'C',
 '(D) radish',
 'A',
 'C',
 'A',
 'D',
 'C',
 'C',
 'C',
 'D',
 'B',
 'A',
 'C',
 'A',
 'C',
 'B',
 'B',
 'D',
 'B',
 'D',
 'A',
 'A',
 'B',
 'B',
 'D',
 'A',
 'ant\n(D)',
 'C',
 'C',
 'A',
 'C',
 'C',
 'D',
 'D',
 'D',
 'A',
 'B',
 'C',
 'D',
 'A',
 'B',
 'D',
 'D',
 'D',
 'A',
 'B',
 'B',
 'C',
 'D',
 'D',
 'B',
 'C',
 'A',
 'ant\n(C)',
 'B',
 'B',
 'A',
 '(A) visibility',
 'C',
 'B',
 'C',
 'C',
 'B',
 'B',
 'B',
 'B',
 'B',
 '(A) phone',
 'B',
 'A',
 'B',
 'A',
 'D',
 'C',
 'C',
 'C',
 'D',
 'D',
 'A',
 'C',
 'B',
 'A',
 'D',
 '(A) skating',
 'C',
 'C',
 'B',
 '(B) turning',
 'B',
 'B',
 'B',
 'C',
 'D',
 'A',
 'C',
 'A',
 'A',
 'D',
 'B',
 'A',
 'B',
 '(B) twenty first',
 'C',
 'A',
 'C',
 'B',
 'C',
 'A',
 'B',
 'D',
 'C',
 'D',
 'A',
 'A',
 'B',
 'B',
 'C',
 'C',
 'B',
 'D',
 'A',
 'D',
 'D',
 'D',
 'A',
 '(B) computer monitor',
 'A',
 'D',
 'B',
 'A',
 'A',
 'D',
 'B',
 'B',
 'D',
 'D',
 'B',
 'A',
 'C',
 '

In [62]:
np.mean(answers == np.array(outputs))

np.float64(0.8183406113537118)

In [35]:
scores, score_arr = calc_metric(answers, outputs, return_scores=True)

/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/nltk/translate/bleu_score.py:577: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it c

IndexError: list index out of range

In [66]:
len(candidates)

1145

In [63]:
import time

start = time.perf_counter()
random_int = np.random.randint(2, size=len(answers))
outputs = []
for i in range(len(candidates)):
    if random_int[i]:
        outputs.append(str(candidates[i][2]))
    else:
        outputs.append(str(candidates[i][3]))
end = time.perf_counter()

print(end - start)
# scores, score_arr = calc_metric(answers, outputs, return_scores=True)

0.003471860196441412


In [67]:
random_int

array([1, 1, 1, ..., 0, 0, 0])

In [68]:
np.mean(answers == np.array(outputs))

np.float64(0.6890829694323144)

In [64]:
np.mean(answers == np.array(outputs))

np.float64(0.2645962732919255)

In [61]:
outputs

['stant\nB',
 'stant\nC',
 'stant\nA',
 ' C',
 ' D',
 ' C',
 ' (B) 3.47',
 'stant\nC',
 'stant\nC',
 ' C',
 ' (C) the Maximum possible loss is unlimited',
 ' A',
 'stant\nB',
 ' B',
 ' C',
 'stant\nB',
 ' (C) $126,925',
 ' C',
 'stant\nB',
 'stant\nB',
 ' C',
 'stant\nB',
 ' B',
 ' 1.2\\)\n',
 ' D',
 'stant\nB',
 ' B',
 'stant\nD',
 ' B',
 ' B',
 'stant\nC',
 'stant\nE',
 ' C',
 ' E',
 'stant\nD',
 ' A',
 'stant\nD',
 'stant\nB',
 ' B',
 'stant\nA',
 'stant\nB',
 'stant\nD',
 'stant\nD',
 ' A',
 ' (B) Downy mildew (oomycete)',
 'stant\nD',
 ' E',
 ' C',
 ' D',
 ' C',
 ' B',
 ' A',
 ' C',
 'stant\nA',
 ' D',
 ' D',
 ' (A) It will grow down into the roots and kill the tree.',
 ' D',
 ' C',
 'stant\nB',
 ' (D) 37.567 $\\approx$ 38',
 ' C',
 '.455 \\)',
 ' D',
 ' C',
 'stant\nA',
 ' A',
 'stant\nD',
 ' C',
 ' C',
 'stant\nA',
 'stant\nB',
 ' C',
 'stant\nB',
 'stant\nB',
 ' C',
 ' C',
 ' C',
 'stant\nD',
 'stant\nD',
 'stant\nD',
 'stant\nB',
 'stant\nD',
 'stant\nC',
 ' C',
 ' C',
 ' C',


In [70]:
candidates

array([['D', 'D', 'D', 'D', 'D'],
       ['D', 'D', 'D', '(D) thirty', 'D'],
       ['C', 'C', 'C', 'B', 'C'],
       ...,
       ['D', 'D', 'A', 'B', 'C'],
       ['D', 'D', 'C', 'B', 'D'],
       ['A', 'A', 'A', 'A', 'A']], dtype=object)

In [69]:
import llm_blender
blender = llm_blender.Blender()
blender.loadranker("llm-blender/PairRM") # load ranker checkpoint


/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/dataclasses_json/core.py:201: RuntimeWarning: 'NoneType' object value of non-optional type load_checkpoint detected when decoding RankerConfig.
  warnings.warn(
/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/dataclasses_json/core.py:201: RuntimeWarning: 'NoneType' object value of non-optional type device detected when decoding RankerConfig.
  warnings.warn(
/home/hice1/stekin6/.conda/envs/v3fusion/lib/python3.9/site-packages/transformers/convert_slow_tokenizer.py:566: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Successfully loaded ranker from  /home/hice1/stekin6/scratch/hf-cache/hub/llm-blender/PairRM


In [73]:
ranks = blender.rank(inputs, candidates.astype(str), return_scores=False, batch_size=32)

Ranking candidates: 100%|██████████| 36/36 [02:09<00:00,  3.61s/it]


In [74]:
ranks

array([[1, 1, 1, 1, 1],
       [1, 4, 1, 5, 3],
       [2, 1, 2, 5, 4],
       ...,
       [4, 4, 1, 3, 2],
       [3, 3, 1, 2, 3],
       [1, 1, 1, 1, 1]], dtype=int32)

In [80]:
outputs = []
for i in range(len(candidates)):
    rank_idx = ranks[i, -1] - 1
    outputs.append(str(candidates[i][rank_idx]))

In [83]:
np.mean(outputs == answers)

np.float64(0.7056768558951965)

In [79]:
rank_idx

array([0, 2, 3, ..., 1, 2, 0], dtype=int32)